In [3]:
import os
import shutil
import pandas as pd
import torch
from PIL import Image
import open_clip

# пути
META_PATH = "./datasets/BCN_org/metadata_org.csv"
IMG_DIR = "./datasets/BCN_org/"
OUT_DIR = "./datasets/bcn_org"

os.makedirs(OUT_DIR, exist_ok=True)

DIAGNOSIS_MAP = {
    "Solar or actinic keratosis": "AK",
    "Basal cell carcinoma": "BCC",
    "Seborrheic keratosis": "BKL",
    "Solar lentigo": "BKL",
    "Dermatofibroma": "DF",
    "Melanoma metastasis": "MEL",
    "Melanoma, NOS": "MEL",
    "Nevus": "NV",
    "Squamous cell carcinoma, NOS": "SCC",
}

PROMPT_MAP = {
    "AK": "Actinic keratosis",
    "BCC": "Basal cell carcinoma",
    "BKL": "Benign keratosis",
    "DF": "Dermatofibroma",
    "MEL": "Melanoma",
    "NV": "Nevus",
    "SCC": "Squamous cell carcinoma",
}


df = pd.read_csv(META_PATH)

df = df[df["diagnosis_3"].isin(DIAGNOSIS_MAP.keys())].copy()

df["class"] = df["diagnosis_3"].map(DIAGNOSIS_MAP)

print("Images after filtering:", len(df))

for c in set(DIAGNOSIS_MAP.values()):
    os.makedirs(os.path.join(OUT_DIR, c), exist_ok=True)


# --- BiomedCLIP ---
device = "cuda" if torch.cuda.is_available() else "cpu"
model, preprocess_train, preprocess_val = open_clip.create_model_and_transforms(
    'hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224'
)
tokenizer = open_clip.get_tokenizer('hf-hub:microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224')
model.to(device)
model.eval()

FEATURES = [

    "rough scaly surface", "pearly border", "visible blood vessels",
    "waxy appearance", "central white scar", "asymmetric shape",
    "irregular border", "multiple colors", "symmetric round shape",
    "uniform brown pigmentation", "blue-white veil", "dot-vessels",
    "symmetric shape",
    
    "arborizing vessels", "comma-shaped vessels", "hairpin vessels", "milky-red pink areas",
    
    "atypical pigment network", "typical pigment network", "peripheral streaks",
    "milia-like cysts", "comedo-like openings", "leaf-like structures", 
    "spoke-wheel areas", "crystalline structures", "ulceration"
]

def get_top_features(image_path, features_list, threshold=0.15, top_n=4):
    image = preprocess_val(Image.open(image_path)).unsqueeze(0).to(device)
    texts = tokenizer(features_list).to(device)
    
    with torch.no_grad(), torch.amp.autocast(device_type=device):
        image_features, text_features, logit_scale = model(image, texts)
        logits_per_image = logit_scale * image_features @ text_features.T
        probs = logits_per_image.softmax(dim=-1).cpu().numpy()[0]
    
    top_indices = probs.argsort()[::-1]
    
    results = [features_list[i] for i in top_indices if probs[i] > threshold]
    
    return results[:top_n]

print("BiomedCLIP...")
print("Starting processing with hybrid prompting...")

print("Starting processing with adaptive confidence thresholds...")

for i, row in df.iterrows():
    img_id = row["isic_id"]
    cls = row["class"] # MEL, NV, DF, BCC и т.д.
    src = os.path.join(IMG_DIR, f"{img_id}.jpg")
    
    if not os.path.exists(src):
        continue

    if cls in ["MEL", "NV", "DF"]:
        current_threshold = 0.4  
    else:
        current_threshold = 0.15 

    detected_features = get_top_features(src, FEATURES, threshold=current_threshold)
    
    trigger = f"dx_{cls.lower()}" 
    class_label = PROMPT_MAP[cls]
    features_str = ", ".join(detected_features)

    if features_str:
        final_prompt = f"{trigger}, {class_label.lower()}, {features_str}, dermoscopy image"
    else:
        final_prompt = f"{trigger}, {class_label.lower()}, dermoscopy image"

    dst_txt = os.path.join(OUT_DIR, cls, f"{img_id}.txt")
    with open(dst_txt, "w") as f:
        f.write(final_prompt)
        
    dst_img = os.path.join(OUT_DIR, cls, f"{img_id}.jpg")
    if not os.path.exists(dst_img):
        shutil.copy2(src, dst_img)

    if i % 100 == 0:
        print(f"Processed {i} images...")

print("Ready! Dataset is prepared with hybrid strategy.")

/opt/conda/envs/lora/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
/opt/conda/envs/lora/lib/python3.10/site-packages/timm/models/layers/__init__.py:48: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)


Images after filtering: 17325


/opt/conda/envs/lora/lib/python3.10/site-packages/open_clip/factory.py:128: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path, map_locati

BiomedCLIP...
Starting processing with hybrid prompting...
Starting processing with adaptive confidence thresholds...
Processed 0 images...
Processed 100 images...
Processed 200 images...
Processed 300 images...
Processed 500 images...
Processed 600 images...
Processed 700 images...
Processed 800 images...
Processed 900 images...
Processed 1000 images...
Processed 1100 images...
Processed 1200 images...
Processed 1300 images...
Processed 1400 images...
Processed 1600 images...
Processed 1700 images...
Processed 1800 images...
Processed 1900 images...
Processed 2000 images...
Processed 2100 images...
Processed 2200 images...
Processed 2300 images...
Processed 2400 images...
Processed 2500 images...
Processed 2600 images...
Processed 2700 images...
Processed 2800 images...
Processed 2900 images...
Processed 3000 images...
Processed 3100 images...
Processed 3200 images...
Processed 3300 images...
Processed 3400 images...
Processed 3700 images...
Processed 3800 images...
Processed 3900 ima